In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import joblib

# import some libraries for training
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.optim as optim
import xgboost as xgb

from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error
import matplotlib.pyplot as plt

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)


In [ ]:
df = pd.read_csv(os.path.join('datasets', 'flextrack_phase1_train.csv'))
print(df.shape)
display(df.head())

In [ ]:
# Try extracting data that has Demand_Response_Flag = 0 and Demand_Response_Capacity_kW != 0.0
df1 = df[
    (df['Demand_Response_Flag']==0) &
    (df['Demand_Response_Capacity_kW'] != 0.0)
].copy(deep=True)
print (df1.shape[0])

df1 = df[
    df['Demand_Response_Flag']==0
].copy(deep=True)
df2 = df1[
    df1['Demand_Response_Capacity_kW'] != 0.0
].copy(deep=True)
print (df2.shape[0])
# We observe that if Demand_Response_Flag = 0, then Demand_Response_Capacity_kW is always 0.0

In [ ]:
def preprocess_data(df_in):
    df = df_in.copy()
    # First, need to remove rows with Demand_Response_Flag = 0
    # df = df[df['Demand_Response_Flag'] != 0].copy()
    # Convert 'Timestamp' to datetime
    df['Timestamp'] = pd.to_datetime(df['Timestamp_Local'])
    # Extract datetime features
    df['Hour'] = df['Timestamp'].dt.hour
    df['Day'] = df['Timestamp'].dt.day
    df['Month'] = df['Timestamp'].dt.month
    df['Weekday'] = df['Timestamp'].dt.weekday
    df['Minute'] = df['Timestamp'].dt.minute
    # Build seasonal features
    df['Is_Weekend'] = df['Weekday'].isin([5, 6]).astype(int)
    df['Is_Summer'] = df['Month'].isin([5, 6, 7, 8]).astype(int)
    df['Is_Winter'] = df['Month'].isin([12, 1, 2, 3]).astype(int)
    # Create hour of day categories
    df['Is_Afternoon'] = df['Hour'].isin(range(12, 18)).astype(int)
    df['Is_Evening'] = df['Hour'].isin(range(18, 24)).astype(int)
    # drop unused columns
    df.drop(columns=['Timestamp_Local','Timestamp','Site'], inplace=True)
    return df

df = preprocess_data(df)
df.head()

In [ ]:
df['Demand_Response_Capacity_kW'].hist(bins=100)

In [ ]:
# Prepare features and target
X = df.drop(columns=['Demand_Response_Capacity_kW']).values
y = df['Demand_Response_Capacity_kW'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=5/X.shape[0], random_state=42)


# labels
is_nz = (y_train != 0).astype(int)
# 1) Zero vs non-zero:
#    Train XGBoost classifier to identify which observations have zero value for 'Demand_Response_Capacity_kW'
#    So we have 2 classes: 0 (zero) and 1 (non-zero) for 'Demand_Response_Capacity_kW'
clf_nz = xgb.XGBClassifier(
    n_estimators=600, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, tree_method="hist",
    eval_metric="logloss",
    # handle class imbalance (zeros >> non-zeros)
    scale_pos_weight=(is_nz==0).sum() / max((is_nz==1).sum(), 1)
).fit(X_train, is_nz)

# 2) Train XGBoost regressor:
#    only on non-zero observations i.e., we will regress 'Demand_Response_Capacity_kW' on non-zero values
mask = is_nz == 1
reg_nz = xgb.XGBRegressor(
    objective='reg:squarederror', 
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    gamma=0.5,
    reg_lambda=1.0,
    reg_alpha=0.5,
    random_state=42
).fit(X_train[mask], y_train[mask])

# Predict on test set
# Inference
p_nz = clf_nz.predict_proba(X_test)[:, 1]
mu_nz = reg_nz.predict(X_test)
y_pred_xgb = p_nz * mu_nz

In [ ]:
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

mae = mean_absolute_error(y_test, y_pred_xgb)
print(f"MAE: {mae:.4f}")

In [ ]:
y_test[:10], y_pred_xgb[:10]

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(y_test, bins=100, alpha=0.5, label='y_test')
plt.hist(y_pred_xgb, bins=100, alpha=0.5, label='y_pred_xgb')
plt.legend()
plt.xlabel('Demand_Response_Capacity_kW')
plt.ylabel('Frequency')
plt.title('Distribution of y_test and y_pred_xgb')
plt.show()

In [ ]:
# Remove zeros in y_test and corresponding entries in y_pred_xgb
mask_nonzero = y_test != 0
y_test_nz = y_test[mask_nonzero]
y_pred_xgb_nz = y_pred_xgb[mask_nonzero]


fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
ax1.hist(y_test, bins=100, alpha=0.5, label='y_test')
ax1.hist(y_pred_xgb, bins=100, alpha=0.5, label='y_pred_xgb')
ax1.legend()
ax1.set_xlabel('Demand_Response_Capacity_kW')
ax1.set_ylabel('Frequency')

ax2.hist(y_test_nz, bins=100, alpha=0.5, label='y_test')
ax2.hist(y_pred_xgb_nz, bins=100, alpha=0.5, label='y_pred_xgb')
ax2.legend()
ax2.set_xlabel('Demand_Response_Capacity_kW')
ax2.set_ylabel('Frequency')

plt.title('Distribution of y_test and y_pred_xgb')
plt.show()

In [ ]:
# Save models
joblib.dump(clf_nz, 'xgb_clf_nz_model.pkl')
joblib.dump(reg_nz, 'xgb_reg_nz_model.pkl')